# Week 7 - Dimensionality Reduction (1/2)
**Notebook 1 : Feature Selection - the 10 metrics of the lecture, on the lecture's data and on real reviews**

**Unstructured Data Analysis (2026-2)** · Professor: Misuk Kim · Teaching Assistant: Hojin Son

A term-document matrix has one column per word. On 2,000 reviews that is already more than ten thousand columns,
almost all of them zero. Lecture 6-2 / 7-1 *Dimensionality Reduction* asks how to make that matrix smaller without
losing what matters. There are two roads: **feature selection** keeps some of the original columns, **feature
extraction** builds new ones. This notebook is the first road; Notebook 2 (LSA and t-SNE) is the second.

| Lecture chapter | What we do in code |
|---|---|
| 1. Dimensionality Reduction | measure how high-dimensional and how sparse our matrix really is; selection vs extraction on a toy matrix |
| 2. Feature Selection, metrics 1-10 | rebuild the lecture's **10-document, 10-term** data set and reproduce every number of its tables (DF, Acc, AccR, PR, OddR, OddN, F1, IG, $\chi^2$, BNS) |
| 2. Feature Selection, applied | compute the same 10 metrics for every word of 2,000 movie reviews and see which words each metric likes |
| 1. Filter vs Wrapper | keep the top-$k$ words of a metric (filter), then let a learning algorithm choose words one by one (wrapper) |

> **How to use this notebook in Colab**: `File ▸ Save a copy in Drive`, then run the cells from top to bottom with
> `Shift + Enter`. Run the **Setup** cell first.

## &nbsp;0. Setup
Only NLTK (for the corpus and the stop-word list), NumPy, pandas, SciPy and scikit-learn. All of them are pre-installed in Colab.

In [ ]:
import nltk
for r in ['movie_reviews', 'stopwords']:
    nltk.download(r, quiet=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.feature_extraction.text import CountVectorizer

%config InlineBackend.figure_format = 'retina'
%matplotlib inline
pd.set_option('display.width', 140)
np.set_printoptions(linewidth=140)

## &nbsp;1. How big is the matrix, really?
Lecture, *Common features of text data*: a document consists of a large number of terms, only a few of them matter,
and the term-document matrix has two problems:

* **Issue 1 - high dimensionality**: the number of terms is much larger than the number of documents
* **Issue 2 - sparseness**: most elements of the matrix are zero

Let us measure both on the corpus we know from Week 4 and Week 6: the 2,000 NLTK movie reviews (1,000 positive, 1,000 negative).
We build a **binary** term-document matrix (1 = the word occurs in the review), because every metric of the lecture
counts *documents*, not occurrences.

In [ ]:
from nltk.corpus import movie_reviews, stopwords

english_stops = set(stopwords.words('english'))
fileids = movie_reviews.fileids()
docs = [' '.join(movie_reviews.words(f)) for f in fileids]
y = np.array([1 if f.startswith('pos') else 0 for f in fileids])     # 1 = Pos, 0 = Neg

vec = CountVectorizer(binary=True, min_df=5, token_pattern=r'[a-z]{2,}', stop_words=list(english_stops))
X = vec.fit_transform(docs)                 # documents x terms, sparse, entries 0 / 1
terms = vec.get_feature_names_out()

print('documents :', X.shape[0], '  (Pos', (y == 1).sum(), '/ Neg', (y == 0).sum(), ')')
print('terms     :', X.shape[1], '  (words in at least 5 reviews, stop words removed)')
print('non-zero  :', X.nnz, f'of {X.shape[0] * X.shape[1]:,} cells  ->  sparsity {1 - X.nnz / (X.shape[0] * X.shape[1]):.1%}')

Six times more columns than rows, and 98 % of the cells are empty: a review uses about 240 of the 13,000 words. This is what the lecture means by
*"N. terms >> N. documents"* and *"Most elements in a term-document matrix are zero"*.

The lecture's taxonomy (slide *A simplified taxonomy*) splits the remedies in two:

| | What it does | Uses the class label? | Uses the learning algorithm? | Examples |
|---|---|---|---|---|
| **Feature selection - filter** | keeps a subset of the original columns, chosen by a statistic | yes | no | the 10 metrics of this notebook |
| **Feature selection - wrapper** | keeps a subset, chosen by training a model again and again | yes | yes | forward / backward selection |
| **Feature extraction** | builds new columns from the old ones | no | no | PCA, MDS, **LSA** (Notebook 2) |

We start with the lecture's small data set, because on 10 documents every number can be checked by hand.

## &nbsp;2. The lecture's artificial data set
Slide *Artificial Data Set*: 10 documents, 10 terms, a binary term-document matrix, 6 positive and 4 negative documents.
We type it in exactly as on the slide. $A$ has one row per **term** and one column per **document**, like the slide.

In [ ]:
A = np.array([
    [1, 1, 1, 1, 1, 1, 0, 0, 0, 0],   # Term 1
    [0, 0, 0, 0, 0, 0, 1, 1, 1, 1],   # Term 2
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],   # Term 3
    [1, 1, 1, 1, 1, 1, 1, 1, 0, 0],   # Term 4
    [0, 0, 0, 1, 1, 1, 1, 1, 1, 1],   # Term 5
    [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],   # Term 6
    [0, 0, 0, 0, 0, 0, 1, 1, 0, 0],   # Term 7
    [1, 0, 1, 0, 1, 0, 1, 0, 1, 0],   # Term 8
    [1, 1, 1, 0, 0, 0, 1, 0, 0, 0],   # Term 9
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 1],   # Term 10
])
toy_terms = [f'Term {i}' for i in range(1, 11)]
toy_docs = [f'D{i}' for i in range(1, 11)]
toy_y = np.array([1, 1, 1, 1, 1, 1, 0, 0, 0, 0])      # Pos = 1, Neg = 0

table = pd.DataFrame(A, index=toy_terms, columns=toy_docs)
table.loc['Class'] = ['Pos' if c else 'Neg' for c in toy_y]
table

### &nbsp;2-1. The four counts every metric is made of
Every formula of the lecture uses the same four numbers for a word $w$:

| symbol | meaning |
|---|---|
| $N(Pos)$, $N(Neg)$ | number of positive / negative documents |
| $N(Pos, w)$, $N(Neg, w)$ | number of positive / negative documents **containing** $w$ |
| $N(Pos, \bar w)$, $N(Neg, \bar w)$ | number of positive / negative documents **not** containing $w$ |
| $N(w) = N(Pos, w) + N(Neg, w)$, $N = N(Pos) + N(Neg)$ | |

`class_counts` computes them for **every term at once** from a documents-by-terms matrix, so the same function works
for the 10 terms here and for the 10,000 terms of Section 3.

In [ ]:
def class_counts(X, y):
    # X: documents x terms (0/1), y: 1 = Pos, 0 = Neg  ->  one value per term
    X = X if isinstance(X, np.ndarray) else X.toarray() if X.shape[1] < 50 else X
    N_pos_w = np.asarray(X[y == 1].sum(axis=0)).ravel().astype(float)
    N_neg_w = np.asarray(X[y == 0].sum(axis=0)).ravel().astype(float)
    return dict(N_pos=float((y == 1).sum()), N_neg=float((y == 0).sum()),
                N_pos_w=N_pos_w, N_neg_w=N_neg_w)


C = class_counts(A.T, toy_y)           # A.T : documents x terms
print('N(Pos) =', C['N_pos'], '  N(Neg) =', C['N_neg'])
print('N(Pos, w) :', C['N_pos_w'])
print('N(Neg, w) :', C['N_neg_w'])

Check with the slide: Term 1 is in 6 positive and 0 negative documents, Term 2 in 0 and 4, Term 3 in 6 and 4.

### &nbsp;2-2. Metrics 1-4 : DF, Acc, AccR, PR
$$DF(w) = N_D(w) \qquad Acc(w) = N(Pos,w) - N(Neg,w)$$
$$AccR(w) = \left| \frac{N(Pos,w)}{N(Pos)} - \frac{N(Neg,w)}{N(Neg)} \right| \qquad PR(w) = \frac{N(Pos,w)}{N(Pos)} \bigg/ \frac{N(Neg,w)}{N(Neg)}$$

$PR$ divides by zero for a word that never occurs in a negative document: the slide writes $\infty$, and so does NumPy.

In [ ]:
def metrics_1_to_4(C):
    N_pos, N_neg, N_pos_w, N_neg_w = C['N_pos'], C['N_neg'], C['N_pos_w'], C['N_neg_w']
    with np.errstate(divide='ignore', invalid='ignore'):
        DF = N_pos_w + N_neg_w
        Acc = N_pos_w - N_neg_w
        AccR = np.abs(N_pos_w / N_pos - N_neg_w / N_neg)
        PR = (N_pos_w / N_pos) / (N_neg_w / N_neg)
    return pd.DataFrame({'DF': DF, 'Acc': Acc, 'AccR': AccR, 'PR': PR})


m14 = metrics_1_to_4(C)
m14.index = toy_terms
m14.round(2)

This is the table of the slide *Compute the metric 1-4 for the data set*, number for number.
Read Term 3: it is in **every** document, so DF is the largest of all, but AccR is 0 and PR is 1: it tells us nothing about the class.
That is the lecture's warning about DF: *"words that appear too often may provide unnecessary information"*.

### &nbsp;2-3. Metrics 5-7 : OddR, OddN, F1
$$OddR(w) = \frac{N(Pos,w)}{N(Neg,w)} \times \frac{N(Neg,\bar w)}{N(Pos,\bar w)} \qquad OddN(w) = N(Pos,w) \times N(Neg,\bar w)$$
$$F1(w) = \frac{2 \times N(Pos,w)}{N(Pos) + N(w)}$$

The slide says for OddR: *"Add 1 to any zero count in the denominator to avoid division by zero"*. We do exactly that:
only a count that is zero becomes 1.

In [ ]:
def metrics_5_to_7(C):
    N_pos, N_neg, N_pos_w, N_neg_w = C['N_pos'], C['N_neg'], C['N_pos_w'], C['N_neg_w']
    N_pos_nw = N_pos - N_pos_w            # N(Pos, not w)
    N_neg_nw = N_neg - N_neg_w            # N(Neg, not w)
    den1 = np.where(N_neg_w == 0, 1, N_neg_w)      # the slide's rule: a zero count in a denominator becomes 1
    den2 = np.where(N_pos_nw == 0, 1, N_pos_nw)
    OddR = (N_pos_w / den1) * (N_neg_nw / den2)
    OddN = N_pos_w * N_neg_nw
    F1 = 2 * N_pos_w / (N_pos + N_pos_w + N_neg_w)
    return pd.DataFrame({'OddR': OddR, 'OddN': OddN, 'F1': F1})


m57 = metrics_5_to_7(C)
m57.index = toy_terms
m57.round(2)

Compare with the slide *Compute the metric 5-7*: Term 8 gives $\frac{3}{2} \times \frac{2}{3} = 1$ and Term 9 gives
$\frac{3}{1} \times \frac{3}{3} = 3$, exactly the two worked examples of the OddR slide. Term 1, with two zero counts
replaced by 1, gives $6 \times 4 = 24$.

Two things to notice:

* **OddR and OddN are one-sided.** A word that lives only in negative documents (Term 2, Term 7) gets 0, not a large value.
  They rank words for the *positive* class. The same is true of Acc and PR, and the lecture says it for F1:
  *"negative features are devalued compared to positive features"*.
* **OddN** is just the numerator of OddR. It is an integer and never infinite, which is why it exists.

### &nbsp;2-4. Metrics 8-10 : IG, $\chi^2$, BNS
These three look at the whole $2 \times 2$ table of (class) $\times$ (word present / absent), so they are **symmetric**:
a word that identifies the negative class scores as high as a word that identifies the positive class.

**Information gain** is the drop in entropy once we know whether $w$ is present:
$$IG(w) = Entropy_{absent}(w) - Entropy_{given}(w)$$
$$Entropy_{absent}(w) = \sum_{C} -P(C) \log P(C), \qquad
Entropy_{given}(w) = P(w) \sum_{C} -P(C|w) \log P(C|w) + P(\bar w) \sum_{C} -P(C|\bar w) \log P(C|\bar w)$$
The slide's worked example gives $Entropy_{absent} = -0.6 \log 0.6 - 0.4 \log 0.4 = 0.29$, which is the value with the
**base-10** logarithm, so we use `np.log10`. The slide also says *"Convert log(0) to zero"*.

**Chi-squared** measures how far the observed $2 \times 2$ table is from independence:
$$\chi^2(w) = \frac{N \left( P(Pos,w) P(Neg,\bar w) - P(Neg,w) P(Pos,\bar w) \right)^2}{P(w) P(\bar w) P(Pos) P(Neg)}$$

**Bi-normal separation** maps the two rates through the inverse normal c.d.f. $F^{-1}$:
$$BNS(w) = \left| F^{-1}\!\left(\frac{N(Pos,w)}{N(Pos)}\right) - F^{-1}\!\left(\frac{N(Neg,w)}{N(Neg)}\right) \right|$$
$F^{-1}(1)$ is infinite, so the slide replaces a rate of 1 by $0.9995$ (and a rate of 0 by $0.0005$): $F^{-1}(0.9995) = 3.29$.

In [ ]:
def entropy_terms(p):
    # -p log10 p, elementwise, with 0 log 0 = 0
    p = np.asarray(p, dtype=float)
    out = np.zeros_like(p)
    m = p > 0
    out[m] = -p[m] * np.log10(p[m])
    return out


def metrics_8_to_10(C):
    N_pos, N_neg, N_pos_w, N_neg_w = C['N_pos'], C['N_neg'], C['N_pos_w'], C['N_neg_w']
    N = N_pos + N_neg
    N_w = N_pos_w + N_neg_w
    N_pos_nw, N_neg_nw = N_pos - N_pos_w, N_neg - N_neg_w
    with np.errstate(divide='ignore', invalid='ignore'):
        # --- IG
        P_w, P_nw = N_w / N, 1 - N_w / N
        E_absent = entropy_terms(N_pos / N) + entropy_terms(N_neg / N)
        E_given = (P_w * (entropy_terms(N_pos_w / N_w) + entropy_terms(N_neg_w / N_w))
                   + P_nw * (entropy_terms(N_pos_nw / (N - N_w)) + entropy_terms(N_neg_nw / (N - N_w))))
        IG = E_absent - np.nan_to_num(E_given)
        # --- chi-squared
        num = N * ((N_pos_w / N) * (N_neg_nw / N) - (N_neg_w / N) * (N_pos_nw / N)) ** 2
        den = P_w * P_nw * (N_pos / N) * (N_neg / N)
        chi2 = np.where(den > 0, num / den, 0.0)        # a word in every (or no) document: 0
        # --- BNS
        tpr = np.clip(N_pos_w / N_pos, 0.0005, 0.9995)
        fpr = np.clip(N_neg_w / N_neg, 0.0005, 0.9995)
        BNS = np.abs(norm.ppf(tpr) - norm.ppf(fpr))
    return pd.DataFrame({'IG': IG, 'chi2': chi2, 'BNS': BNS})


m810 = metrics_8_to_10(C)
m810.index = toy_terms
m810.round(2)

The slide *Compute the metric 8-10*: Term 1 has $IG = 0.29$ (knowing Term 1 removes all uncertainty), $\chi^2 = 10.00$ and
$BNS = 3.29 - (-3.29) = 6.58$; Term 4 has $\chi^2 = 3.75$ and $BNS = 3.29$, the two worked examples. Term 2 gets the same
scores as Term 1: these three metrics do not care which class the word points to.

### &nbsp;2-5. All ten metrics, and which words they agree on
The lecture's summary slide puts the ten columns side by side and counts, for every term, in how many of the ten rankings
it is among the **top 3**. Ties share a rank (Term 1 and Term 2 are both first for IG, so the next term is third).

In [ ]:
def all_metrics(X, y, names):
    C = class_counts(X, y)
    M = pd.concat([metrics_1_to_4(C), metrics_5_to_7(C), metrics_8_to_10(C)], axis=1)
    M.index = names
    return M


def top3_count(M, k=3):
    # rank every column from the largest value down; ties share the best rank (as on the slide)
    ranks = M.round(2).rank(ascending=False, method='min')
    return (ranks <= k).sum(axis=1)


M_toy = all_metrics(A.T, toy_y, toy_terms)
summary = M_toy.round(2)
summary['Top3'] = top3_count(M_toy)
summary

This is the slide *Comparison of the 10 feature selection metrics*: Term 4 is in the top 3 of all ten metrics, Term 1 of nine,
Term 6 of six. The two words that only mark the negative class, Term 2 and Term 7, are found by the symmetric metrics
(AccR, IG, $\chi^2$, BNS) and by nothing else.

### &nbsp;2-6. Selection keeps columns, extraction makes new ones
The slide *Feature selection vs. feature extraction*, on our matrix. $V_1 \ldots V_{10}$ are the ten terms (one column each,
documents as rows).

In [ ]:
V = A.T                                    # documents x terms : V1 ... V10 are the columns

# Feature selection: keep columns 1, 5 and 8 (nothing new is computed)
selected = V[:, [0, 4, 7]]

# Feature extraction: three NEW columns, each a combination of the old ones (as written on the slide)
Z1 = V[:, 0] + 0.2 * V[:, 1]               # Z1 = V1 + 0.2 * V2
Z2 = V[:, 2] - 2 * V[:, 4]                 # Z2 = V3 - 2 * V5
Z3 = V[:, 3] + V[:, 5] - V[:, 8]           # Z3 = V4 + V6 - V9
extracted = np.column_stack([Z1, Z2, Z3])

print('selection  (V1, V5, V8) :\n', selected[:4])
print('\nextraction (Z1, Z2, Z3) :\n', extracted[:4])

The selected matrix still contains only 0 and 1 and keeps the names of the words. The extracted one has new values
(0.2, -2, ...) that no longer belong to one word. Notebook 2 builds the $Z$ columns in a principled way, with SVD.

## &nbsp;3. The same ten metrics on 2,000 reviews
`all_metrics` works on any documents-by-terms matrix, so we give it the sparse matrix of Section 1
(2,000 reviews, Pos = 1). One line, ten metrics, every word.

In [ ]:
M = all_metrics(X, y, terms)
print(M.shape)
M.loc[['film', 'bad', 'wonderful', 'boring', 'waste', 'perfectly']].round(3)

`film` is the Term 3 of this corpus: the highest DF, and IG, $\chi^2$ and BNS all about 0. `bad` and `waste` belong to
the negative class, so Acc is negative, PR is below 1, OddR is small, but IG, $\chi^2$ and BNS are large.

Now the words each metric likes best.

In [ ]:
def top_words(M, k=10):
    return pd.DataFrame({col: M[col].sort_values(ascending=False).index[:k] for col in M.columns},
                        index=[f'#{i}' for i in range(1, k + 1)])


top_words(M, 10)

Read the columns in the three groups of the lecture:

* **DF** returns the words of the *topic* (one, film, movie, like): frequent, and useless for the class.
* **Acc, PR, OddR, OddN, F1** return *positive* words only. PR and OddR, which divide by the negative count, put words
  that are **never** negative on top, however rare they are (`obscurity`, `avoids`: five reviews each, $PR = \infty$).
  OddN and F1, which multiply instead of dividing, prefer words that are frequent *and* positive (`life`, `also`, `best`).
* **AccR, IG, $\chi^2$** return the words that *separate* the classes, in both directions: `bad`, `worst`, `stupid`,
  `boring`, `waste`, `ridiculous` next to `life`, `great`, `best`. IG and $\chi^2$ give almost the same list.
* **BNS** is symmetric too, but its top is made of rare words seen in one class only (`lovingly`, `melancholy`, `gattaca`):
  a rate of 0 is clipped to 0.0005, and $F^{-1}(0.0005) = -3.29$ is a long way from $F^{-1}(5/1000) = -2.58$.
  On ten documents that did not matter; on two thousand it decides the ranking.

As on the lecture's summary slide, we count in how many of the ten top-20 lists a word appears.

In [ ]:
agreement = top3_count(M, k=20).sort_values(ascending=False)
print(agreement.head(12).to_string())

No word is in the top 20 of more than six metrics. The lecture's Term 4 (top 3 of all ten) has no counterpart here: with
13,000 words and two groups of metrics that look for different things (frequent positive words vs separating words),
the agreement is low. The words most metrics agree on (`life`, `well`, `also`, `best`) are the frequent positive ones,
because six of the ten metrics grow with the positive count.

## &nbsp;4. Filter vs Wrapper
Slide *Filter approach vs. Wrapper approach*:

* **Filter** : rank the words by a metric, keep the top $k$. No feedback loop, independent of the learning algorithm. That is what we did above.
* **Wrapper** : evaluate a subset by training a learning algorithm, keep what improves it, repeat. Slower, and the result depends on the algorithm.

To compare them we need a learning algorithm that returns one number. We use scikit-learn's `LogisticRegression` as a
**black box**: it is trained on 1,500 reviews and reports its accuracy on the other 500. How it works is the topic of
Week 10; today it is only a measuring instrument.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(np.arange(X.shape[0]), test_size=500, stratify=y, random_state=7)


def score(columns):
    # accuracy of a logistic regression that sees only the given columns (a black box, see Week 10)
    cols = np.asarray(columns)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(X[train_idx][:, cols], y[train_idx])
    return clf.score(X[test_idx][:, cols], y[test_idx])


print(f'all {X.shape[1]} words : accuracy {score(np.arange(X.shape[1])):.3f}')

### &nbsp;4-1. Filter : the top-$k$ words of a metric

In [ ]:
ks = [20, 100, 500, 2000, X.shape[1]]
rows = {}
for metric in ['DF', 'PR', 'F1', 'IG', 'chi2', 'BNS']:
    order = M[metric].sort_values(ascending=False).index
    rows[metric] = [score([vec.vocabulary_[w] for w in order[:k]]) for k in ks]

filter_table = pd.DataFrame(rows, index=[f'top {k}' for k in ks[:-1]] + ['all']).T
filter_table.round(3)

With only **100 words** chosen by IG or $\chi^2$ the black box is about as accurate as with all 13,037 (0.83 vs 0.85);
with **500** it is *better* (0.89), and BNS with 2,000 words gives the best number of the table (0.91). The same 100 words
chosen by DF (frequent words) or PR (rare, purely positive words) are far worse. The ranking, not the number, is what
matters. This is the lecture's *"Improve classification accuracy"* and *"Reduce the amount of training data needed"* in
one table. (The numbers can differ in the last digit between scikit-learn versions.)

> scikit-learn has `SelectKBest(chi2)` for this job. Its `chi2` is **not** the lecture's formula: it compares the observed and
> expected counts of the word's *presence* only, and ignores the absent-row of the $2 \times 2$ table. The rankings are
> close, the numbers are not. The assignment uses the lecture's formula, as above.

### &nbsp;4-2. Wrapper : let the learning algorithm choose
Greedy **forward selection**: start from no word, try every candidate, keep the one that raises the accuracy most, repeat.
We give it the 100 best $\chi^2$ words as candidates (10,000 candidates would take all afternoon) and let it pick 10.
A wrapper must never look at the test reviews while choosing, so it chooses on a validation split cut from the training reviews.

In [ ]:
tr_idx, val_idx = train_test_split(train_idx, test_size=300, stratify=y[train_idx], random_state=7)


def val_score(columns):
    cols = np.asarray(columns)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(X[tr_idx][:, cols], y[tr_idx])
    return clf.score(X[val_idx][:, cols], y[val_idx])


candidates = [vec.vocabulary_[w] for w in M['chi2'].sort_values(ascending=False).index[:100]]
chosen, curve = [], []
for step in range(10):
    best = max((c for c in candidates if c not in chosen), key=lambda c: val_score(chosen + [c]))
    chosen.append(best)
    curve.append(val_score(chosen))
    print(f'step {step + 1:>2}: + {terms[best]:<12} validation accuracy {curve[-1]:.3f}')

print('\nwrapper, 10 words :', [terms[c] for c in chosen])
print('filter,  10 words :', list(M['chi2'].sort_values(ascending=False).index[:10]))
print(f'\ntest accuracy  wrapper {score(chosen):.3f}   filter top-10 chi2 {score(candidates[:10]):.3f}')

The wrapper trained the black box about a thousand times to choose ten words; the filter sorted one column.
The wrapper's ten words are **not** the ten best $\chi^2$ words: after `bad` is in, a second word that says the same thing
(`worst`, `stupid`) adds less than a word that says something new (`true`, `supposed`). That feedback loop is the whole
difference between the two approaches.

And on the 500 test reviews the wrapper's ten words are **no better** than the filter's ten. It chose the words that fit
its 300 validation reviews, which is the wrapper's weakness: the choice depends on the learning algorithm *and* on the
data it was tuned on. The lecture's *"can offer higher accuracy but may take longer"* is a *can*, not a promise, and
the *longer* is certain.

## Summary

* A term-document matrix is wide and empty: 13,000 columns for 2,000 reviews, 98 % zeros.
* The ten metrics are ten ways of reading the same four counts $N(Pos,w)$, $N(Neg,w)$, $N(Pos)$, $N(Neg)$.
  DF ignores the class; Acc, PR, OddR, OddN and F1 favor the positive class; AccR, IG, $\chi^2$ and BNS are symmetric.
* Filter = rank and cut, once. Wrapper = train, compare, repeat. A good filter with a few hundred words loses nothing,
  and can even gain.
* Notebook 2 takes the other road: instead of keeping columns, it builds new ones with SVD (LSA) and draws them with t-SNE.